# Demo-ready sketch → photo v2: artifact-free pix2pix experts

This notebook replaces the rejected blurry-refiner experiments. It trains two independent
deployable models at 256×256:

1. **Portrait expert** — real paired FS2K artist sketches and photographs.
2. **General-scene expert** — persistent COCO photos with aligned synthetic line art.

Both use a standard skip-connected pix2pix U-Net and a 70×70 conditional PatchGAN. The
experts save/resume separately in Drive, so Colab limits do not erase progress. The existing
photo→pencil model and all previous runs remain untouched.

Honest limitation: a monochrome sketch does not contain exact colour information. These
models aim for plausible, sharper demo output—not recovery of the original photograph.

V2 repair: all decoder upsampling uses bilinear resize followed by reflection-padded convolution, removing the checkerboard mechanism seen in the rejected v1 step-1500 gate. The paired pixel weight is reduced and perceptual/edge/adversarial weights are increased to discourage blur. V1 checkpoints are intentionally incompatible and are not resumed.


In [ ]:
# Install only the small packages missing from a normal Colab GPU runtime.
# Do not replace Colab's torch/torchvision wheels.
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'lpips==0.1.4', 'gdown==5.2.0',
])
print('Dependencies ready.')


In [ ]:
from pathlib import Path
from collections import defaultdict
from copy import deepcopy
import hashlib, json, math, os, random, shutil, time, zipfile

import cv2, gdown, lpips
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Choose Runtime → Change runtime type → GPU.'
DEVICE = torch.device('cuda')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

from google.colab import drive
drive.mount('/content/drive')

SAVE_ROOT = Path('/content/drive/MyDrive/Sketch2Photo_training')
COCO_ROOT = SAVE_ROOT / 'datasets/coco2017_general_20k'
FS2K_STORAGE = SAVE_ROOT / 'datasets/FS2K_persistent'
RUN_ROOT = SAVE_ROOT / 'demo_pix2pix_experts_v2_resizeconv'
RUN_ROOT.mkdir(parents=True, exist_ok=True); FS2K_STORAGE.mkdir(parents=True, exist_ok=True)

FS2K_SAMPLE_PROBABILITY = .5  # compatibility with the shared preprocessing cell
VALIDATION_PER_DOMAIN = 20    # compatibility with the shared preprocessing cell
PORTRAIT_UPDATES = 8_000
GENERAL_UPDATES = 10_000
SAVE_EVERY = 250
VALIDATE_EVERY = 500
RESOLUTION = 256
print('GPU:', torch.cuda.get_device_name(0))
print('Run folder:', RUN_ROOT)
print('COCO will NOT be downloaded again.')

## 1. Discover the persistent paired datasets

In [ ]:
ALLOWED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}

def atomic_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, default=str))
    os.replace(temporary, path)

def atomic_torch(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    torch.save(value, temporary)
    os.replace(temporary, path)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_extract(zip_path, destination):
    destination = Path(destination); destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for entry in archive.infolist():
            resolved = (destination / entry.filename).resolve()
            if not resolved.is_relative_to(root) or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
                raise ValueError('Unsafe ZIP entry: ' + entry.filename)
        archive.extractall(destination)

def find_fs2k_root():
    candidates = list(FS2K_STORAGE.rglob('anno_train.json'))
    if not candidates:
        archive = FS2K_STORAGE / 'FS2K.zip'
        if not archive.is_file():
            print('FS2K is absent from Drive; downloading it once to persistent storage...')
            gdown.download(
                id='1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp',
                output=str(archive), quiet=False,
            )
        if not zipfile.is_zipfile(archive):
            raise ValueError('The downloaded FS2K file is not a valid ZIP.')
        safe_extract(archive, FS2K_STORAGE / 'extracted')
        candidates = list(FS2K_STORAGE.rglob('anno_train.json'))
    assert len(candidates) == 1, f'Expected one FS2K root, found {len(candidates)}.'
    root = candidates[0].parent
    assert (root / 'anno_test.json').is_file(), 'FS2K test annotations are missing.'
    return root

def normalize_stem(value):
    return Path(str(value).replace('\\', '/')).with_suffix('').as_posix().casefold()

def image_index(root, folder):
    base = root / folder
    index = defaultdict(list)
    for path in base.rglob('*'):
        if path.is_file() and path.suffix.lower() in ALLOWED_EXTENSIONS:
            index[normalize_stem(path.relative_to(base))].append(path)
    assert index, f'No images found in {base}'
    return index

def build_fs2k_records(root):
    train_rows = json.loads((root / 'anno_train.json').read_text())
    test_rows = json.loads((root / 'anno_test.json').read_text())
    grouped = defaultdict(list)
    for row in train_rows:
        grouped[str(row['style'])].append(row)
    tagged = []
    for style, rows in sorted(grouped.items()):
        rows = sorted(rows, key=lambda row: row['image_name'])
        random.Random(SEED + int(hashlib.sha256(style.encode()).hexdigest()[:8], 16)).shuffle(rows)
        cut = max(1, round(0.2 * len(rows)))
        tagged += [(row, 'val') for row in rows[:cut]]
        tagged += [(row, 'train') for row in rows[cut:]]
    tagged += [(row, 'test') for row in test_rows]

    photos, sketches = image_index(root, 'photo'), image_index(root, 'sketch')
    records, excluded = [], []
    for row, split in tagged:
        photo_stem = Path(str(row['image_name']).replace('\\', '/')).with_suffix('').as_posix()
        parts = photo_stem.split('/')
        parts[0] = parts[0].replace('photo', 'sketch', 1)
        sketch_path = Path('/'.join(parts))
        name = sketch_path.name
        if name.startswith('image'):
            name = 'sketch' + name[len('image'):]
        sketch_stem = (sketch_path.parent / name).as_posix()
        photo_matches = photos.get(normalize_stem(photo_stem), [])
        sketch_matches = sketches.get(normalize_stem(sketch_stem), [])
        if len(photo_matches) != 1 or len(sketch_matches) != 1:
            excluded.append(str(row['image_name']))
            continue
        records.append({
            'id': 'fs2k_' + photo_stem,
            'split': split,
            'domain': 'fs2k_artist',
            'photo': str(photo_matches[0]),
            'sketch': str(sketch_matches[0]),
        })
    assert len(records) >= int(0.98 * len(tagged)), (
        f'Only {len(records)}/{len(tagged)} FS2K pairs were resolved. Check the archive.'
    )
    return records, excluded

def build_coco_records():
    manifest_path = COCO_ROOT / 'manifest.json'
    photo_root = COCO_ROOT / 'photos'
    assert manifest_path.is_file(), (
        f'Missing {manifest_path}. Copy/share the completed dataset folder; this notebook will not redownload COCO.'
    )
    manifest = json.loads(manifest_path.read_text())
    records = []
    for row in manifest.get('records', []):
        filename = row.get('file_name') or Path(row.get('photo', '')).name
        photo = photo_root / filename
        if photo.is_file():
            records.append({
                'id': str(row.get('id', 'coco_' + photo.stem)),
                'split': row['split'],
                'domain': 'coco_synthetic',
                'photo': str(photo),
                'sketch': '',
            })
    train_count = sum(row['split'] == 'train' for row in records)
    assert train_count >= 20_000, (
        f'Only {train_count}/20000 persistent COCO training photos were found. '
        'Do not train on an incomplete folder.'
    )
    return records

FS2K_ROOT = find_fs2k_root()
FS2K_RECORDS, FS2K_EXCLUDED = build_fs2k_records(FS2K_ROOT)
COCO_RECORDS = build_coco_records()

FS2K_SPLITS = {split: [r for r in FS2K_RECORDS if r['split'] == split] for split in ('train','val','test')}
COCO_SPLITS = {split: [r for r in COCO_RECORDS if r['split'] == split] for split in ('train','val','test')}
for split in ('train', 'val', 'test'):
    print(split, '| FS2K:', len(FS2K_SPLITS[split]), '| COCO:', len(COCO_SPLITS[split]))

DATA_FINGERPRINT = hashlib.sha256(json.dumps({
    'fs2k': [(r['id'], r['split']) for r in FS2K_RECORDS],
    'coco': [(r['id'], r['split']) for r in COCO_RECORDS],
    'seed': SEED,
}, sort_keys=True).encode()).hexdigest()
atomic_json(RUN_ROOT / 'data_audit.json', {
    'fs2k_root': str(FS2K_ROOT),
    'fs2k_usable': len(FS2K_RECORDS),
    'fs2k_excluded': FS2K_EXCLUDED,
    'coco_root': str(COCO_ROOT),
    'coco_usable': len(COCO_RECORDS),
    'fingerprint': DATA_FINGERPRINT,
})
print('Data fingerprint:', DATA_FINGERPRINT)


## 2. Shared paired preprocessing

In [ ]:
def fit_canvas(image, size):
    image = ImageOps.exif_transpose(image).convert('RGB')
    scale = min(size / image.width, size / image.height)
    width, height = max(1, round(image.width * scale)), max(1, round(image.height * scale))
    resized = image.resize((width, height), Image.Resampling.LANCZOS)
    canvas = Image.new('RGB', (size, size), 'white')
    left, top = (size - width) // 2, (size - height) // 2
    canvas.paste(resized, (left, top))
    return canvas, (left, top, left + width, top + height)

def make_clean_line_art(photo):
    rgb = np.asarray(photo.convert('RGB'))
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    smooth = cv2.bilateralFilter(gray, 7, 32, 7)
    inverted = 255 - smooth
    blurred = cv2.GaussianBlur(inverted, (0, 0), 5.0)
    dodge = cv2.divide(smooth, np.maximum(255 - blurred, 1).astype(np.uint8), scale=256)
    fine = cv2.GaussianBlur(smooth, (0, 0), .65).astype(np.float32)
    medium = cv2.GaussianBlur(smooth, (0, 0), 1.65).astype(np.float32)
    edges = np.maximum(medium - fine, 0) / 255.0
    shadow = np.power((255 - smooth.astype(np.float32)) / 255.0, 1.45)
    density = .74 * ((255 - dodge.astype(np.float32)) / 255.0) + .22 * shadow + 2.1 * edges
    drawing = np.clip(254 - 255 * (1 - np.exp(-2.05 * density)), 8, 255).astype(np.uint8)
    return Image.fromarray(drawing).convert('RGB')

def augment_sketch(image, rng):
    sketch = ImageOps.autocontrast(image.convert('L'), cutoff=.2)
    sketch = ImageEnhance.Contrast(sketch).enhance(rng.uniform(.88, 1.15))
    sketch = ImageEnhance.Brightness(sketch).enhance(rng.uniform(.97, 1.03))
    if rng.random() < .15:
        sketch = sketch.filter(ImageFilter.MinFilter(3))
    elif rng.random() < .15:
        sketch = sketch.filter(ImageFilter.GaussianBlur(rng.uniform(.15, .45)))
    array = np.asarray(sketch, dtype=np.float32)
    noise = np.random.default_rng(rng.randrange(2**32)).normal(0, rng.uniform(0, 1.0), array.shape)
    return Image.fromarray(np.clip(array + noise, 0, 255).astype(np.uint8)).convert('RGB')

def tensor_image(image):
    array = np.asarray(image, dtype=np.float32).copy() / 127.5 - 1.0
    return torch.from_numpy(array.transpose(2, 0, 1))

def load_pair(record, resolution):
    with Image.open(record['photo']) as image:
        photo, box = fit_canvas(image, resolution)
    if record['domain'] == 'fs2k_artist':
        with Image.open(record['sketch']) as image:
            sketch, sketch_box = fit_canvas(image, resolution)
        assert sketch_box == box, 'FS2K pair geometry does not match.'
        sketch = sketch.convert('L').convert('RGB')
    else:
        sketch = make_clean_line_art(photo)
    return photo, sketch, box

class FocusedDataset(Dataset):
    def __init__(self, total_indices, resolution, training=True):
        self.total_indices = total_indices
        self.resolution = resolution
        self.training = training

    def __len__(self):
        return len(self.total_indices)

    def __getitem__(self, position):
        global_index = self.total_indices[position]
        rng = random.Random(SEED + 1_000_003 * global_index)
        use_fs2k = rng.random() < FS2K_SAMPLE_PROBABILITY
        pool = FS2K_SPLITS['train'] if use_fs2k else COCO_SPLITS['train']
        record = pool[rng.randrange(len(pool))]
        photo, sketch, _ = load_pair(record, self.resolution)
        if self.training and rng.random() < .5:
            photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
        if self.training and rng.random() < .35:
            sketch = augment_sketch(sketch, rng)
        return {
            'source': tensor_image(sketch),
            'target': tensor_image(photo),
            'domain': record['domain'],
            'id': record['id'],
        }

VALIDATION_RECORDS = (
    FS2K_SPLITS['val'][:VALIDATION_PER_DOMAIN]
    + COCO_SPLITS['val'][:VALIDATION_PER_DOMAIN]
)
assert len(VALIDATION_RECORDS) == 2 * VALIDATION_PER_DOMAIN

fig, axes = plt.subplots(4, 2, figsize=(9, 15))
preview_records = FS2K_SPLITS['train'][:2] + COCO_SPLITS['train'][:2]
for row, record in enumerate(preview_records):
    photo, sketch, box = load_pair(record, 512)
    for ax, image, title in zip(axes[row], [sketch.crop(box), photo.crop(box)], ['Sketch input', 'Photo target']):
        ax.imshow(image); ax.set_title(record['domain'] + ' — ' + title); ax.axis('off')
plt.tight_layout()
plt.savefig(RUN_ROOT / 'pair_audit.png', dpi=130)
plt.show()
print('STOP if any row above is mismatched. Otherwise continue.')


In [ ]:
def paired_train_crop(record, seed):
    rng = random.Random(seed)
    photo, sketch, _ = load_pair(record, 286)
    if rng.random() < .5:
        photo, sketch = ImageOps.mirror(photo), ImageOps.mirror(sketch)
    left, top = rng.randrange(31), rng.randrange(31)
    box = (left, top, left+256, top+256)
    photo, sketch = photo.crop(box), sketch.crop(box)
    if rng.random() < .25:
        sketch = augment_sketch(sketch, rng)
    return tensor_image(sketch), tensor_image(photo)

class ExpertDataset(Dataset):
    def __init__(self, expert, start, stop):
        self.expert, self.start, self.stop = expert, start, stop
        self.pool = FS2K_SPLITS['train'] if expert == 'portrait' else COCO_SPLITS['train']
    def __len__(self): return self.stop-self.start
    def __getitem__(self, position):
        step = self.start+position
        rng = random.Random(SEED + step*1_000_003 + (0 if self.expert=='portrait' else 17))
        record = self.pool[rng.randrange(len(self.pool))]
        source, target = paired_train_crop(record, rng.randrange(2**31))
        return {'source':source, 'target':target, 'id':record['id']}

VAL_RECORDS = {
    'portrait': FS2K_SPLITS['val'][:20],
    'general': COCO_SPLITS['val'][:20],
}
TEST_RECORDS = {
    'portrait': FS2K_SPLITS['test'],
    'general': COCO_SPLITS['test'],
}
print('Portrait train/val/test:', len(FS2K_SPLITS['train']), len(VAL_RECORDS['portrait']), len(TEST_RECORDS['portrait']))
print('General train/val/test:', len(COCO_SPLITS['train']), len(VAL_RECORDS['general']), len(TEST_RECORDS['general']))

## 3. Standard pix2pix architecture

This generator differs from the previous blurry model: each decoder level directly receives
the corresponding high-resolution encoder features. A strong conditional PatchGAN teaches
local photographic texture instead of relying almost entirely on pixel averages.

In [ ]:
class Down(nn.Module):
    def __init__(self, a, b, norm=True):
        super().__init__(); layers=[nn.Conv2d(a,b,4,2,1,bias=not norm)]
        if norm: layers.append(nn.InstanceNorm2d(b, affine=True))
        layers.append(nn.LeakyReLU(.2, inplace=True)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

class Up(nn.Module):
    def __init__(self, a, b, dropout=False):
        super().__init__(); layers=[nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
            nn.ReflectionPad2d(1),nn.Conv2d(a,b,3,bias=False),
            nn.InstanceNorm2d(b,affine=True),nn.ReLU(inplace=True)]
        if dropout: layers.append(nn.Dropout(.35))
        self.net=nn.Sequential(*layers)
    def forward(self,x,skip): return torch.cat([self.net(x),skip],1)

class Pix2PixGenerator(nn.Module):
    def __init__(self, width=64):
        super().__init__()
        self.d1=Down(3,width,False); self.d2=Down(width,width*2); self.d3=Down(width*2,width*4)
        self.d4=Down(width*4,width*8); self.d5=Down(width*8,width*8); self.d6=Down(width*8,width*8)
        self.d7=Down(width*8,width*8); self.d8=Down(width*8,width*8,False)
        self.u1=Up(width*8,width*8,True); self.u2=Up(width*16,width*8,True)
        self.u3=Up(width*16,width*8,True); self.u4=Up(width*16,width*8)
        self.u5=Up(width*16,width*4); self.u6=Up(width*8,width*2); self.u7=Up(width*4,width)
        self.out=nn.Sequential(nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False),
            nn.ReflectionPad2d(3),nn.Conv2d(width*2,3,7),nn.Tanh())
    def forward(self,x):
        d1=self.d1(x); d2=self.d2(d1); d3=self.d3(d2); d4=self.d4(d3)
        d5=self.d5(d4); d6=self.d6(d5); d7=self.d7(d6); d8=self.d8(d7)
        u1=self.u1(d8,d7); u2=self.u2(u1,d6); u3=self.u3(u2,d5); u4=self.u4(u3,d4)
        u5=self.u5(u4,d3); u6=self.u6(u5,d2); u7=self.u7(u6,d1)
        return self.out(u7)

class PatchGAN(nn.Module):
    def __init__(self,width=64):
        super().__init__(); layers=[nn.Conv2d(6,width,4,2,1),nn.LeakyReLU(.2,True)]
        for a,b,stride in [(width,width*2,2),(width*2,width*4,2),(width*4,width*8,1)]:
            layers += [nn.Conv2d(a,b,4,stride,1,bias=False),nn.InstanceNorm2d(b,affine=True),nn.LeakyReLU(.2,True)]
        layers += [nn.Conv2d(width*8,1,4,1,1)]; self.net=nn.Sequential(*layers)
    def forward(self,source,image): return self.net(torch.cat([source,image],1))

def init_pix2pix(module):
    if isinstance(module,nn.Conv2d):
        nn.init.normal_(module.weight,0,.02)
        if module.bias is not None: nn.init.zeros_(module.bias)

PERCEPTUAL = lpips.LPIPS(net='alex',version='0.1').to(DEVICE).eval().requires_grad_(False)

def gradients(x): return x[:,:,:,1:]-x[:,:,:,:-1], x[:,:,1:,:]-x[:,:,:-1,:]

def generator_losses(fake,target):
    fake,target=fake.float(),target.float(); fx,fy=gradients(fake); tx,ty=gradients(target)
    with torch.autocast('cuda',enabled=False): perceptual=PERCEPTUAL(fake,target).mean()
    l1=F.l1_loss(fake,target); edge=F.l1_loss(fx,tx)+F.l1_loss(fy,ty)
    colour=F.l1_loss(fake.mean((2,3)),target.mean((2,3)))
    return {'l1':l1,'lpips':perceptual,'edge':edge,'colour':colour,
            'paired':45*l1+10*perceptual+15*edge+8*colour}

def to_image(tensor):
    a=((tensor.detach().float().cpu()[0].clamp(-1,1).permute(1,2,0).numpy()+1)*127.5).round().astype('uint8')
    return Image.fromarray(a)

## 4. Resumable expert trainer

The portrait and general models have independent state folders. Each validation produces a
four-column visual gate and preserves the lowest LPIPS/MAE checkpoint. Non-finite gradients
are skipped safely instead of corrupting Drive state.

In [ ]:
@torch.no_grad()
def validate_expert(expert,model):
    model.eval(); rows=[]
    for record in VAL_RECORDS[expert]:
        photo,sketch,_=load_pair(record,256)
        source=tensor_image(sketch)[None].to(DEVICE); target=tensor_image(photo)[None].to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16): prediction=model(source)
        with torch.autocast('cuda',enabled=False): perceptual=float(PERCEPTUAL(prediction.float(),target.float()).mean())
        mae=float(F.l1_loss(prediction.float(),target.float())/2)
        rows.append({'id':record['id'],'lpips':perceptual,'mae':mae})
    return {'lpips':float(np.mean([r['lpips'] for r in rows])),
            'mae':float(np.mean([r['mae'] for r in rows])),'score':float(np.mean([r['lpips']+.25*r['mae'] for r in rows]))}

@torch.no_grad()
def visual_gate(expert,model,step,path):
    model.eval(); records=VAL_RECORDS[expert][:4]
    fig,axes=plt.subplots(4,3,figsize=(10,13))
    for row,record in enumerate(records):
        photo,sketch,box=load_pair(record,256); source=tensor_image(sketch)[None].to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16): prediction=model(source)
        for ax,image,title in zip(axes[row],[sketch.crop(box),to_image(prediction).crop(box),photo.crop(box)],
                                  ['Input sketch',f'{expert} output step {step}','Real target']):
            ax.imshow(image); ax.set_title(title); ax.axis('off')
    fig.tight_layout(); fig.savefig(path,dpi=140); plt.show(); plt.close(fig)

def cpu_state(model): return {k:v.detach().cpu() for k,v in model.state_dict().items()}

def train_expert(expert,total_updates):
    root=RUN_ROOT/expert; root.mkdir(exist_ok=True)
    generator=Pix2PixGenerator().to(DEVICE); discriminator=PatchGAN().to(DEVICE)
    generator.apply(init_pix2pix); discriminator.apply(init_pix2pix)
    ema=deepcopy(generator).eval().requires_grad_(False)
    opt_g=torch.optim.Adam(generator.parameters(),lr=2e-4,betas=(.5,.999))
    opt_d=torch.optim.Adam(discriminator.parameters(),lr=2e-4,betas=(.5,.999))
    scaler=torch.amp.GradScaler('cuda',init_scale=128,growth_interval=2000)
    progress={'step':0,'best_step':0,'best_score':float('inf'),'seconds':0.0}; history=[]
    last=root/'last.pt'; signature=f'pix2pix-demo-v2-resizeconv-{expert}-{total_updates}'
    if last.is_file():
        state=torch.load(last,map_location='cpu',weights_only=False)
        assert state['signature']==signature
        generator.load_state_dict(state['generator']); discriminator.load_state_dict(state['discriminator'])
        ema.load_state_dict(state['ema']); opt_g.load_state_dict(state['opt_g']); opt_d.load_state_dict(state['opt_d'])
        scaler.load_state_dict(state['scaler']); progress,history=state['progress'],state['history']
        print('Resumed',expert,'at',progress['step'])
    started=time.perf_counter(); elapsed=progress['seconds']
    def save():
        progress['seconds']=elapsed+time.perf_counter()-started
        atomic_torch(last,{'signature':signature,'generator':generator.state_dict(),'discriminator':discriminator.state_dict(),
            'ema':ema.state_dict(),'opt_g':opt_g.state_dict(),'opt_d':opt_d.state_dict(),'scaler':scaler.state_dict(),
            'progress':dict(progress),'history':history})
        atomic_json(root/'history.json',history); atomic_json(root/'progress.json',progress)
    def evaluate():
        metrics=validate_expert(expert,ema); history.append({'type':'validation','step':progress['step'],**metrics})
        if metrics['score']<progress['best_score']:
            progress['best_score']=metrics['score']; progress['best_step']=progress['step']
            atomic_torch(root/'best.pt',{'format':'sketch2photo-demo-pix2pix-v2-resizeconv','expert':expert,
                'direction':'sketch_to_photo','resolution':256,'step':progress['step'],
                'validation':metrics,'width':64,'generator':cpu_state(ema)})
        visual_gate(expert,ema,progress['step'],root/f"gate_{progress['step']:07d}.png")
        print(expert,'validation:',metrics,'best step:',progress['best_step'])
    if not history: evaluate(); save()
    bar=tqdm(total=total_updates,initial=progress['step'],desc=expert+' pix2pix')
    while progress['step']<total_updates:
        boundary=min(total_updates,progress['step']+SAVE_EVERY)
        loader=DataLoader(ExpertDataset(expert,progress['step'],boundary),batch_size=1,shuffle=False,
                          num_workers=2,pin_memory=True)
        generator.train(); discriminator.train()
        for batch in loader:
            source=batch['source'].to(DEVICE,non_blocking=True); target=batch['target'].to(DEVICE,non_blocking=True)
            with torch.autocast('cuda',dtype=torch.float16): fake=generator(source)
            opt_d.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                real_score=discriminator(source,target); fake_score=discriminator(source,fake.detach())
                loss_d=.5*(F.relu(1-real_score).mean()+F.relu(1+fake_score).mean())
            scaler.scale(loss_d).backward(); scaler.unscale_(opt_d); nn.utils.clip_grad_norm_(discriminator.parameters(),5)
            scaler.step(opt_d)
            discriminator.requires_grad_(False); opt_g.zero_grad(set_to_none=True)
            terms=generator_losses(fake,target)
            with torch.autocast('cuda',dtype=torch.float16): adversarial=-discriminator(source,fake).mean()
            loss_g=terms['paired']+2.0*adversarial.float()
            scaler.scale(loss_g).backward(); scaler.unscale_(opt_g)
            norm=nn.utils.clip_grad_norm_(generator.parameters(),5,error_if_nonfinite=False)
            if not torch.isfinite(norm):
                opt_g.zero_grad(set_to_none=True); discriminator.requires_grad_(True)
                scaler.update(new_scale=max(float(scaler.get_scale())*.5,1)); print('Skipped unsafe batch'); continue
            scaler.step(opt_g); scaler.update(); discriminator.requires_grad_(True)
            with torch.no_grad():
                decay=.99 if progress['step']<2000 else .999
                for e,p in zip(ema.parameters(),generator.parameters()): e.lerp_(p,1-decay)
            progress['step']+=1; step=progress['step']
            history.append({'type':'train','step':step,'generator':float(loss_g.detach()),
                'discriminator':float(loss_d.detach()),'l1':float(terms['l1']),'lpips':float(terms['lpips'])})
            bar.update(); bar.set_postfix(g=round(float(loss_g.detach()),2),d=round(float(loss_d.detach()),2))
            if step%VALIDATE_EVERY==0 or step==total_updates: evaluate()
        save(); del loader; torch.cuda.empty_cache()
    bar.close(); return root/'best.pt'

## 5. Train the portrait expert

Run this cell first. Inspect gates every 500 updates. If Colab disconnects, rerun setup
cells and this cell; it resumes automatically. Do not judge the random step-0 image.

In [ ]:
PORTRAIT_BEST = train_expert('portrait', PORTRAIT_UPDATES)
print('Portrait model:', PORTRAIT_BEST)

## 6. Train the general-scene expert

Run after the portrait expert, or in another Colab session. It has an independent resume
file, so completing one model is never lost while training the other.

In [ ]:
GENERAL_BEST = train_expert('general', GENERAL_UPDATES)
print('General model:', GENERAL_BEST)

## 7. Learning curves, final visual gates, and deployable export

In [ ]:
models={}; summaries={}
fig,axes=plt.subplots(2,2,figsize=(13,8))
for row,expert in enumerate(('portrait','general')):
    root=RUN_ROOT/expert; best=torch.load(root/'best.pt',map_location='cpu',weights_only=True)
    model=Pix2PixGenerator(best['width']).to(DEVICE); model.load_state_dict(best['generator']); model.eval()
    models[expert]=model; summaries[expert]=best['validation']
    history=json.loads((root/'history.json').read_text())
    train=[x for x in history if x['type']=='train']; val=[x for x in history if x['type']=='validation']
    axes[row,0].plot([x['step'] for x in train],[x['generator'] for x in train],alpha=.35)
    axes[row,0].set(title=expert+' generator loss',xlabel='Updates',ylabel='Loss'); axes[row,0].grid(alpha=.2)
    axes[row,1].plot([x['step'] for x in val],[x['lpips'] for x in val],marker='o',label='LPIPS ↓')
    axes[row,1].plot([x['step'] for x in val],[x['mae'] for x in val],marker='o',label='MAE ↓')
    axes[row,1].axvline(best['step'],color='green',linestyle='--',label='selected')
    axes[row,1].set(title=expert+' validation',xlabel='Updates'); axes[row,1].legend(); axes[row,1].grid(alpha=.2)
    visual_gate(expert,model,best['step'],root/'FINAL_GATE.png')
fig.tight_layout(); fig.savefig(RUN_ROOT/'learning_curves.png',dpi=160); plt.show()

manifest={'format':'sketch2photo-demo-pix2pix-experts-v2-resizeconv','direction':'sketch_to_photo','resolution':256,
          'experts':{'portrait':'portrait_best.pt','general':'general_best.pt'},'validation':summaries,
          'note':'Two separately trained pix2pix generators; exact colours are not recoverable from monochrome input.'}
export=RUN_ROOT/'export'; export.mkdir(exist_ok=True)
shutil.copy2(RUN_ROOT/'portrait/best.pt',export/'portrait_best.pt')
shutil.copy2(RUN_ROOT/'general/best.pt',export/'general_best.pt')
shutil.copy2(RUN_ROOT/'portrait/FINAL_GATE.png',export/'portrait_FINAL_GATE.png')
shutil.copy2(RUN_ROOT/'general/FINAL_GATE.png',export/'general_FINAL_GATE.png')
shutil.copy2(RUN_ROOT/'learning_curves.png',export/'learning_curves.png')
atomic_json(export/'manifest.json',manifest)
archive=shutil.make_archive(str(RUN_ROOT/'demo_pix2pix_experts_v2_export'),'zip',export)
print('Portrait:',summaries['portrait']); print('General:',summaries['general'])
print('DOWNLOAD THIS FOR THE VS CODE DEMO:',archive)

## Demo handoff

Download `demo_pix2pix_experts_v2_export.zip` only after both final visual gates are acceptable.
The ZIP contains two directly deployable generators and their manifest. Copy it to the VS
Code project; do not rename individual files. Keep all Drive resume folders until the demo
has been tested locally.